## Preprocesamiento de los datos

In [7]:
# ============================================================
# URBANING-V2X
# PREPROCESAMIENTO DE NUBES LiDAR
#
# Pipeline:
#   LiDAR raw
#       ↓
#   Fusión de sensores
#       ↓
#   ROI
#       ↓
#   Eliminación de suelo (RANSAC)
#       ↓
#   Eliminación de outliers
#       ↓
#   Voxelización
#       ↓
#   Nube procesada
#
# NO utiliza Ground Truth para el procesamiento.
# NO realiza DBSCAN todavía.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors

import urbaning.data.sequence as sequence_module
from urbaning.data import Sequence


# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

ROOT = r"datasets/urbaning-v2x"

SCENARIOS = [
    "20241126_0008_crossing1_01",
    "20241126_0024_crossing1_09",
    "20241127_0000_crossing1_00",
]


# Carpeta donde guardaremos los datos procesados
OUTPUT_ROOT = os.path.join(
    ROOT,
    "processed_lidar"
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)


# ============================================================
# 3. CONFIGURACIÓN DEL PREPROCESAMIENTO
# ============================================================

# ------------------------------------------------------------
# ROI
# ------------------------------------------------------------

ROI_X = (-40.0, 40.0)
ROI_Y = (-40.0, 40.0)
ROI_Z = (-3.0, 5.0)


# ------------------------------------------------------------
# RANSAC - SUELO
# ------------------------------------------------------------

RANSAC_ITERATIONS = 100

RANSAC_DISTANCE_THRESHOLD = 0.15

GROUND_MIN_POINTS = 100


# ------------------------------------------------------------
# OUTLIERS
# ------------------------------------------------------------

OUTLIER_K = 8

OUTLIER_STD_RATIO = 2.0


# ------------------------------------------------------------
# VOXELIZACIÓN
# ------------------------------------------------------------

VOXEL_SIZE = 0.10


# ============================================================
# 4. SEQUENCE SIN GROUND TRUTH
# ============================================================

class SequenceNoGT(Sequence):
    """
    Versión de Sequence que no carga ni procesa
    los archivos de Ground Truth.
    """

    def __init__(self, root_folder, sequence_name):

        self.sequence_root = os.path.join(
            root_folder,
            "dataset",
            sequence_name
        )

        self.sequence_name = sequence_name

        # ----------------------------------------------------
        # Calibration
        # ----------------------------------------------------

        calibration_file = os.path.join(
            self.sequence_root,
            "calibration.json"
        )

        with open(
            calibration_file,
            "r"
        ) as f:

            calib_data = json.load(f)

        self.calib_data = (
            sequence_module.redo_calib(
                calib_data
            )
        )

        # ----------------------------------------------------
        # Time synchronization
        # ----------------------------------------------------

        time_sync_info_file = os.path.join(
            self.sequence_root,
            "timesync_info.csv"
        )

        self.time_sync_df = (
            pd.read_csv(
                time_sync_info_file
            )
            .set_index("Unnamed: 0")
        )

        # ----------------------------------------------------
        # AV vehicle track IDs
        # ----------------------------------------------------

        labels_av_track_ids_file = os.path.join(
            root_folder,
            "labels_av_track_ids.json"
        )

        with open(
            labels_av_track_ids_file,
            "r"
        ) as f:

            self.labels_av_track_ids = (
                json.load(f)[sequence_name]
            )

        # ----------------------------------------------------
        # AV vehicle data
        # ----------------------------------------------------

        av_vehicle_data_file = os.path.join(
            root_folder,
            "av_vehicle_data.json"
        )

        with open(
            av_vehicle_data_file,
            "r"
        ) as f:

            av_vehicle_data = json.load(f)

        self.av_vehicle_data = (
            sequence_module.redo_av_vehicle_data(
                av_vehicle_data
            )
        )

        # ----------------------------------------------------
        # IMPORTANTE:
        # NO CARGAMOS GROUND TRUTH
        # ----------------------------------------------------

        self.labels = {}

        # ----------------------------------------------------
        # World coordinates
        # ----------------------------------------------------

        crossing_name = sequence_name.split("_")[2]

        self.WORLD = "world_coordinates"

        sequence_module._xTg_registry[
            self.WORLD
        ] = sequence_module.get_wTg(
            crossing_name
        )

        # ----------------------------------------------------
        # Lanelet map
        # ----------------------------------------------------

        lanelet_map = os.path.join(
            root_folder,
            "crossings_lanelet2map.osm"
        )

        self.lanelet_map = sequence_module.LLMap(
            lanelet_map,
            sequence_module.gps_origins[
                crossing_name
            ],
            sequence_module.ground_params[
                crossing_name
            ]
        )


# ============================================================
# 5. ROI
# ============================================================

def crop_roi(
    points,
    x_range,
    y_range,
    z_range
):
    """
    Elimina puntos que están fuera de la región de interés.
    """

    if len(points) == 0:
        return points

    mask = (
        (points[:, 0] >= x_range[0]) &
        (points[:, 0] <= x_range[1]) &
        (points[:, 1] >= y_range[0]) &
        (points[:, 1] <= y_range[1]) &
        (points[:, 2] >= z_range[0]) &
        (points[:, 2] <= z_range[1])
    )

    return points[mask]


# ============================================================
# 6. RANSAC PARA EL PLANO DEL SUELO
# ============================================================

def fit_plane_from_points(
    p1,
    p2,
    p3
):
    """
    Calcula el plano que pasa por tres puntos.

    Devuelve:
        a, b, c, d

    para:

        ax + by + cz + d = 0
    """

    v1 = p2 - p1
    v2 = p3 - p1

    normal = np.cross(
        v1,
        v2
    )

    norm = np.linalg.norm(normal)

    if norm < 1e-8:
        return None

    normal = normal / norm

    a, b, c = normal

    d = -np.dot(
        normal,
        p1
    )

    return np.array([
        a,
        b,
        c,
        d
    ])


def remove_ground_ransac(
    points,
    iterations=100,
    distance_threshold=0.15,
    min_points=100
):
    """
    Detecta el plano del suelo mediante RANSAC.

    Devuelve:
        points_without_ground,
        ground_points,
        plane
    """

    if len(points) < min_points:
        return (
            points,
            np.empty((0, points.shape[1])),
            None
        )

    xyz = points[:, :3]

    best_plane = None
    best_inliers = None
    best_score = -1

    n_points = len(xyz)

    for _ in range(iterations):

        # Seleccionar tres puntos aleatorios
        indices = np.random.choice(
            n_points,
            size=3,
            replace=False
        )

        p1 = xyz[indices[0]]
        p2 = xyz[indices[1]]
        p3 = xyz[indices[2]]

        plane = fit_plane_from_points(
            p1,
            p2,
            p3
        )

        if plane is None:
            continue

        a, b, c, d = plane

        # Queremos un plano aproximadamente horizontal.
        #
        # El vector normal debería apuntar
        # principalmente en Z.
        #
        if abs(c) < 0.7:
            continue

        distances = (
            np.abs(
                a * xyz[:, 0]
                + b * xyz[:, 1]
                + c * xyz[:, 2]
                + d
            )
            /
            np.sqrt(
                a*a +
                b*b +
                c*c
            )
        )

        inliers = (
            distances <
            distance_threshold
        )

        score = np.sum(inliers)

        if score > best_score:

            best_score = score
            best_plane = plane
            best_inliers = inliers

    # No se encontró ningún plano
    if best_plane is None:
        return (
            points,
            np.empty((0, points.shape[1])),
            None
        )

    ground_points = points[
        best_inliers
    ]

    non_ground_points = points[
        ~best_inliers
    ]

    return (
        non_ground_points,
        ground_points,
        best_plane
    )


# ============================================================
# 7. ELIMINACIÓN DE OUTLIERS
# ============================================================

def remove_statistical_outliers(
    points,
    k=8,
    std_ratio=2.0
):
    """
    Elimina puntos estadísticamente aislados.
    """

    if len(points) <= k:
        return points

    xyz = points[:, :3]

    neighbors = NearestNeighbors(
        n_neighbors=k + 1,
        algorithm="kd_tree",
        n_jobs=-1
    )

    neighbors.fit(xyz)

    distances, _ = neighbors.kneighbors(
        xyz
    )

    # La primera distancia es 0
    mean_distances = (
        distances[:, 1:]
        .mean(axis=1)
    )

    threshold = (
        mean_distances.mean()
        +
        std_ratio *
        mean_distances.std()
    )

    mask = (
        mean_distances <
        threshold
    )

    return points[mask]


# ============================================================
# 8. VOXEL DOWNSAMPLING
# ============================================================

def voxel_downsample(
    points,
    voxel_size=0.10
):
    """
    Reduce la densidad de la nube mediante voxels.

    Se conserva el primer punto encontrado
    en cada voxel.
    """

    if len(points) == 0:
        return points

    xyz = points[:, :3]

    voxel_coordinates = np.floor(
        xyz / voxel_size
    ).astype(np.int32)

    _, unique_indices = np.unique(
        voxel_coordinates,
        axis=0,
        return_index=True
    )

    unique_indices = np.sort(
        unique_indices
    )

    return points[
        unique_indices
    ]


# ============================================================
# 9. PIPELINE COMPLETO
# ============================================================

def preprocess_lidar(
    points
):
    """
    Ejecuta todo el pipeline de preprocesamiento.

    Devuelve un diccionario con todas las etapas.
    """

    result = {}

    # --------------------------------------------------------
    # RAW
    # --------------------------------------------------------

    result["raw"] = points.copy()

    # --------------------------------------------------------
    # ROI
    # --------------------------------------------------------

    points_roi = crop_roi(
        points,
        ROI_X,
        ROI_Y,
        ROI_Z
    )

    result["roi"] = points_roi

    # --------------------------------------------------------
    # GROUND REMOVAL
    # --------------------------------------------------------

    (
        points_no_ground,
        ground_points,
        plane
    ) = remove_ground_ransac(
        points_roi,
        iterations=RANSAC_ITERATIONS,
        distance_threshold=RANSAC_DISTANCE_THRESHOLD,
        min_points=GROUND_MIN_POINTS
    )

    result["no_ground"] = points_no_ground

    result["ground"] = ground_points

    result["ground_plane"] = plane

    # --------------------------------------------------------
    # OUTLIERS
    # --------------------------------------------------------

    points_no_outliers = (
        remove_statistical_outliers(
            points_no_ground,
            k=OUTLIER_K,
            std_ratio=OUTLIER_STD_RATIO
        )
    )

    result["no_outliers"] = (
        points_no_outliers
    )

    # --------------------------------------------------------
    # VOXELIZATION
    # --------------------------------------------------------

    points_voxel = voxel_downsample(
        points_no_outliers,
        voxel_size=VOXEL_SIZE
    )

    result["voxel"] = points_voxel

    return result


# ============================================================
# 10. ESTADÍSTICAS
# ============================================================

def calculate_statistics(
    processed
):
    """
    Calcula estadísticas de reducción
    de puntos.
    """

    raw = len(
        processed["raw"]
    )

    roi = len(
        processed["roi"]
    )

    no_ground = len(
        processed["no_ground"]
    )

    no_outliers = len(
        processed["no_outliers"]
    )

    voxel = len(
        processed["voxel"]
    )

    def reduction(n):
        if raw == 0:
            return 0.0

        return (
            100 *
            (1 - n / raw)
        )

    return {

        "raw_points": raw,

        "roi_points": roi,
        "roi_reduction_%":
            reduction(roi),

        "no_ground_points":
            no_ground,
        "ground_removed_%":
            reduction(no_ground),

        "no_outliers_points":
            no_outliers,
        "outliers_removed_%":
            reduction(no_outliers),

        "voxel_points":
            voxel,
        "total_reduction_%":
            reduction(voxel)
    }


# ============================================================
# 11. VISUALIZACIÓN BEV
# ============================================================

def plot_bev_comparison(
    processed,
    title=""
):
    """
    Compara RAW vs nube procesada en BEV.
    """

    raw = processed["raw"]

    final = processed["voxel"]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6)
    )

    # --------------------------------------------------------
    # RAW
    # --------------------------------------------------------

    axes[0].scatter(
        raw[:, 0],
        raw[:, 1],
        s=0.2
    )

    axes[0].set_title(
        f"RAW ({len(raw):,} puntos)"
    )

    axes[0].set_xlabel("X [m]")
    axes[0].set_ylabel("Y [m]")

    axes[0].set_aspect(
        "equal",
        adjustable="box"
    )

    axes[0].grid(
        True,
        alpha=0.3
    )

    # --------------------------------------------------------
    # PROCESSED
    # --------------------------------------------------------

    axes[1].scatter(
        final[:, 0],
        final[:, 1],
        s=1
    )

    axes[1].set_title(
        f"PROCESADO ({len(final):,} puntos)"
    )

    axes[1].set_xlabel("X [m]")
    axes[1].set_ylabel("Y [m]")

    axes[1].set_aspect(
        "equal",
        adjustable="box"
    )

    axes[1].grid(
        True,
        alpha=0.3
    )

    fig.suptitle(
        title
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 12. GUARDAR NUBE
# ============================================================

def save_processed_frame(
    output_folder,
    frame_id,
    processed
):
    """
    Guarda la nube procesada y las etapas intermedias.
    """

    os.makedirs(
        output_folder,
        exist_ok=True
    )

    np.save(
        os.path.join(
            output_folder,
            f"frame_{frame_id:06d}_raw.npy"
        ),
        processed["raw"]
    )

    np.save(
        os.path.join(
            output_folder,
            f"frame_{frame_id:06d}_roi.npy"
        ),
        processed["roi"]
    )

    np.save(
        os.path.join(
            output_folder,
            f"frame_{frame_id:06d}_noground.npy"
        ),
        processed["no_ground"]
    )

    np.save(
        os.path.join(
            output_folder,
            f"frame_{frame_id:06d}_nooutliers.npy"
        ),
        processed["no_outliers"]
    )

    np.save(
        os.path.join(
            output_folder,
            f"frame_{frame_id:06d}_processed.npy"
        ),
        processed["voxel"]
    )


# ============================================================
# 13. FUNCIÓN PARA EXTRAER LiDAR DEL FRAME
# ============================================================

def obtener_puntos_globales(frame):
    """
    Fusiona el LiDAR del vehículo ego + todos los LiDARs
    de la infraestructura en coordenadas globales.

    No utiliza Ground Truth.
    """

    all_points = []

    # ============================================================
    # 1. LiDAR del vehículo
    # ============================================================
    vehicle = frame.vehicles["vehicle1"]

    if "vehicle1_middle_lidar" in vehicle.lidars:
        lidar = vehicle.lidars["vehicle1_middle_lidar"]

        points = lidar.point_cloud_in_global_coordinates()

        if points is not None and len(points) > 0:
            all_points.append(np.asarray(points)[:, :3])

    # ============================================================
    # 2. LiDARs de infraestructura
    # ============================================================
    if "crossing1" in frame.infrastructures:

        infrastructure = frame.infrastructures["crossing1"]

        for lidar_name, lidar in infrastructure.lidars.items():

            points = lidar.point_cloud_in_global_coordinates()

            if points is not None and len(points) > 0:
                all_points.append(np.asarray(points)[:, :3])

    # ============================================================
    # 3. Fusionar
    # ============================================================
    if not all_points:
        return np.empty((0, 3), dtype=np.float32)

    points_global = np.vstack(all_points)

    # Eliminar NaN / infinitos
    valid = np.isfinite(points_global).all(axis=1)
    points_global = points_global[valid]

    return points_global.astype(np.float32)


# ============================================================
# 14. PROCESAR ESCENARIOS
# ============================================================

all_statistics = []


for scenario_name in SCENARIOS:

    print("\n")
    print("=" * 70)
    print(
        f"Procesando escenario: {scenario_name}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Cargar secuencia SIN GT
    # --------------------------------------------------------

    sequence = SequenceNoGT(
        ROOT,
        scenario_name
    )

    frame = sequence[0]

    vehicle = frame.vehicles["vehicle1"]

    lidar = vehicle.lidars["vehicle1_middle_lidar"]

    print(type(lidar))
    print("\nATRIBUTOS Y MÉTODOS:")
    print(dir(lidar))

    scenario_output = os.path.join(
        OUTPUT_ROOT,
        scenario_name
    )

    os.makedirs(
        scenario_output,
        exist_ok=True
    )

    print(
        f"Número de frames: {len(sequence)}"
    )

    # --------------------------------------------------------
    # Procesar frames
    # --------------------------------------------------------

    for frame_id, frame in enumerate(
        sequence
    ):

        start_time = time.perf_counter()

        print(
            f"\rFrame {frame_id + 1}/{len(sequence)}",
            end=""
        )

        # ----------------------------------------------------
        # Obtener nube fusionada
        # ----------------------------------------------------

        points = obtener_puntos_globales(
            frame
        )

        if len(points) == 0:

            print(
                f"\nFrame {frame_id}: "
                "sin puntos LiDAR"
            )

            continue

        # ----------------------------------------------------
        # Preprocesamiento
        # ----------------------------------------------------

        processed = preprocess_lidar(
            points
        )

        # ----------------------------------------------------
        # Estadísticas
        # ----------------------------------------------------

        stats = calculate_statistics(
            processed
        )

        stats["scenario"] = (
            scenario_name
        )

        stats["frame"] = (
            frame_id
        )

        stats["processing_time_s"] = (
            time.perf_counter()
            - start_time
        )

        all_statistics.append(
            stats
        )

        # ----------------------------------------------------
        # Guardar datos
        # ----------------------------------------------------

        save_processed_frame(
            scenario_output,
            frame_id,
            processed
        )


    print("\n")
    print(
        f"Escenario terminado: "
        f"{scenario_name}"
    )


# ============================================================
# 15. CREAR DATAFRAME DE ESTADÍSTICAS
# ============================================================

statistics_df = pd.DataFrame(
    all_statistics
)

statistics_path = os.path.join(
    OUTPUT_ROOT,
    "preprocessing_statistics.csv"
)

statistics_df.to_csv(
    statistics_path,
    index=False
)


print("\n")
print("=" * 70)
print("PREPROCESAMIENTO TERMINADO")
print("=" * 70)

print(
    f"\nEstadísticas guardadas en:\n"
    f"{statistics_path}"
)

print("\nResumen:")

display(
    statistics_df.head()
)



Procesando escenario: 20241126_0008_crossing1_01
<class 'urbaning.data.lidar_data.LidarData'>

ATRIBUTOS Y MÉTODOS:
['LIDAR', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_lTg', '_point_cloud_in_global_coordinates', '_point_cloud_in_lidar_coordinates', 'add_coordinates', 'gTl', 'lTg', 'lidar_file_path', 'lidar_index', 'lidar_name', 'point_cloud_in_camera_coordinates', 'point_cloud_in_global_coordinates', 'point_cloud_in_image_coordinates', 'point_cloud_in_infrastructure_coordinates', 'point_cloud_in_lidar_coordinates', 'point_cloud_in_other_lidar_coordinates', 'point_cloud_in_this_coordinates', 'point_cloud_in_vehicle_body_coordin

Frame 200/200

Escenario terminado: 20241126_0008_crossing1_01


Procesando escenario: 20241126_0024_crossing1_09
<class 'urbaning.data.lidar_data.LidarData'>

ATRIBUTOS Y MÉTODOS:
['LIDAR', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_lTg', '_point_cloud_in_global_coordinates', '_point_cloud_in_lidar_coordinates', 'add_coordinates', 'gTl', 'lTg', 'lidar_file_path', 'lidar_index', 'lidar_name', 'point_cloud_in_camera_coordinates', 'point_cloud_in_global_coordinates', 'point_cloud_in_image_coordinates', 'point_cloud_in_infrastructure_coordinates', 'point_cloud_in_lidar_coordinates', 'point_cloud_in_other_lidar_coordinates', 'point_c

,raw_points,roi_points,roi_reduction_%,no_ground_points,ground_removed_%,no_outliers_points,outliers_removed_%,voxel_points,total_reduction_%,scenario,frame,processing_time_s
0,299710,246863,17.632712,166554,44.428281,160502,46.447566,66880,77.685096,20241126_0008_crossing1_01,0,1.287323
1,299434,247517,17.338378,168882,43.599591,163488,45.400990,69714,76.718075,20241126_0008_crossing1_01,1,1.288702
2,299300,247895,17.175075,191582,35.989977,185722,37.947878,83881,71.974273,20241126_0008_crossing1_01,2,1.448190
3,299481,247877,17.231143,178975,40.238279,172631,42.356610,72437,75.812489,20241126_0008_crossing1_01,3,1.376630
4,299882,247402,17.500217,176684,41.082159,170639,43.097952,70482,76.496755,20241126_0008_crossing1_01,4,1.266571


## Clustering y entrenameinto de modelo

In [14]:
import json
import os
import numpy as np


class UrbanIngGTLoader:
    """
    Carga el Ground Truth directamente desde el JSON de UrbanIng-V2X,
    evitando Sequence() y su conversión problemática de orientations.
    """

    def __init__(self, root_folder, sequence_name):

        self.root_folder = root_folder
        self.sequence_name = sequence_name

        self.label_file = os.path.join(
            root_folder,
            "labels",
            sequence_name + ".json"
        )

        with open(self.label_file, "r") as f:
            self.data = json.load(f)

        self.tracks = self.data["tracks"]

        # Índice:
        # timestamp -> lista de objetos
        self.timestamp_to_objects = {}

        self._build_timestamp_index()

    def _build_timestamp_index(self):

        for track in self.tracks:

            track_id = track["track_id"]
            object_type = track["object_type"]

            dimensions = np.asarray(
                track["dimensions"][0],
                dtype=np.float32
            )

            timestamps = track["timestamps"]
            positions = np.asarray(
                track["positions"],
                dtype=np.float32
            )

            orientations = np.asarray(
                track["orientations"],
                dtype=np.float32
            )

            for i, timestamp in enumerate(timestamps):

                if i >= len(positions):
                    continue

                obj = {
                    "track_id": track_id,
                    "object_type": object_type,
                    "dimensions": dimensions.copy(),
                    "position": positions[i].copy(),
                    "orientation": float(orientations[i])
                }

                if timestamp not in self.timestamp_to_objects:
                    self.timestamp_to_objects[timestamp] = []

                self.timestamp_to_objects[timestamp].append(obj)

    def get_objects(self, timestamp):

        return self.timestamp_to_objects.get(
            timestamp,
            []
        )

    def get_vehicle_objects(self, timestamp):

        objects = self.get_objects(timestamp)

        vehicles = []

        for obj in objects:

            object_type = str(
                obj["object_type"]
            ).lower()

            if object_type in {
                "car",
                "vehicle",
                "truck",
                "bus",
                "van",
                "motorcycle"
            }:
                vehicles.append(obj)

        return vehicles

In [33]:
# ============================================================
# URBANING - DBSCAN + FEATURES + RANDOM FOREST
# ============================================================
#
# Pipeline:
#
#   LiDAR preprocesado (.npy)
#          |
#          v
#       DBSCAN
#          |
#          v
#    Cluster features
#          |
#          v
#   GT -> etiquetas 0/1
#          |
#          v
#    Random Forest
#          |
#          v
#       Evaluación
#
# ============================================================

import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import joblib

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

ROOT = r"datasets/UrbanIng-V2X"

PROCESSED_ROOT = os.path.join(
    ROOT,
    "processed_lidar"
)

SCENARIOS = [
    "20241126_0008_crossing1_01",
    "20241126_0024_crossing1_09",
    "20241127_0000_crossing1_00",
]


# ------------------------------------------------------------
# División por escenarios
# ------------------------------------------------------------
#
# IMPORTANTE:
# No dividimos aleatoriamente los clusters porque los frames
# consecutivos son muy similares.
#
# Entrenamos con unos escenarios y evaluamos en otro.
# ------------------------------------------------------------

TRAIN_SCENARIOS = [
    SCENARIOS[0],
    SCENARIOS[1],
]

TEST_SCENARIOS = [
    SCENARIOS[2],
]


# ------------------------------------------------------------
# DBSCAN
# ------------------------------------------------------------

DBSCAN_EPS = 0.7
DBSCAN_MIN_SAMPLES = 8


# ------------------------------------------------------------
# Filtro de clusters
# ------------------------------------------------------------

MIN_CLUSTER_POINTS = 10
MAX_CLUSTER_POINTS = 5000

MIN_HEIGHT = 0.5
MAX_HEIGHT = 4.0

MIN_LENGTH = 0.5
MAX_LENGTH = 15.0

MIN_WIDTH = 0.3
MAX_WIDTH = 8.0


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

RANDOM_STATE = 42
N_ESTIMATORS = 300


# ------------------------------------------------------------
# Probabilidad mínima para considerar vehículo
# ------------------------------------------------------------

VEHICLE_THRESHOLD = 0.5


# ============================================================
# 2. FEATURES UTILIZADAS POR RANDOM FOREST
# ============================================================

FEATURE_COLUMNS = [

    "n_points",

    "centroid_x",
    "centroid_y",
    "centroid_z",

    "length",
    "width",
    "height",

    "volume",
    "density",

    "pca_1",
    "pca_2",
    "pca_3",

    "orientation",

    "height_width_ratio",
    "length_width_ratio",
    "height_length_ratio",
]


# ============================================================
# 3. DBSCAN
# ============================================================

def run_dbscan(
    points,
    eps=DBSCAN_EPS,
    min_samples=DBSCAN_MIN_SAMPLES
):
    """
    Ejecuta DBSCAN sobre una nube de puntos XYZ.

    Returns
    -------
    labels : np.ndarray
        -1 = ruido
        >= 0 = ID del cluster
    """

    if points is None or len(points) == 0:
        return np.empty(0, dtype=int)

    db = DBSCAN(
        eps=eps,
        min_samples=min_samples,
        metric="euclidean",
        n_jobs=-1
    )

    labels = db.fit_predict(points)

    return labels


# ============================================================
# 4. FEATURES DE UN CLUSTER
# ============================================================

def calculate_cluster_features(points):

    if points is None or len(points) < 3:
        return None

    points = np.asarray(points, dtype=np.float64)

    # --------------------------------------------------------
    # Bounding box
    # --------------------------------------------------------

    min_xyz = points.min(axis=0)
    max_xyz = points.max(axis=0)

    dimensions = max_xyz - min_xyz

    dx, dy, dz = dimensions

    # Longitud y anchura en XY
    length = max(dx, dy)
    width = min(dx, dy)

    height = dz

    volume = dx * dy * dz

    # --------------------------------------------------------
    # Centroide
    # --------------------------------------------------------

    centroid = points.mean(axis=0)

    # --------------------------------------------------------
    # Densidad
    # --------------------------------------------------------

    if volume > 1e-6:
        density = len(points) / volume
    else:
        density = 0.0

    # --------------------------------------------------------
    # PCA
    # --------------------------------------------------------

    centered = points - centroid

    covariance = np.cov(
        centered,
        rowvar=False
    )

    eigenvalues, eigenvectors = np.linalg.eigh(
        covariance
    )

    # Orden descendente
    order = np.argsort(
        eigenvalues
    )[::-1]

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    eigenvalues = np.maximum(
        eigenvalues,
        0
    )

    total_variance = (
        np.sum(eigenvalues) + 1e-8
    )

    pca_1 = eigenvalues[0] / total_variance
    pca_2 = eigenvalues[1] / total_variance
    pca_3 = eigenvalues[2] / total_variance

    # --------------------------------------------------------
    # Orientación principal
    # --------------------------------------------------------

    principal_direction = eigenvectors[:, 0]

    orientation = np.arctan2(
        principal_direction[1],
        principal_direction[0]
    )

    # --------------------------------------------------------
    # Ratios
    # --------------------------------------------------------

    height_width_ratio = (
        height / (width + 1e-6)
    )

    length_width_ratio = (
        length / (width + 1e-6)
    )

    height_length_ratio = (
        height / (length + 1e-6)
    )

    # --------------------------------------------------------
    # Resultado
    # --------------------------------------------------------

    return {

        "n_points": len(points),

        "centroid_x": centroid[0],
        "centroid_y": centroid[1],
        "centroid_z": centroid[2],

        "length": length,
        "width": width,
        "height": height,

        "volume": volume,
        "density": density,

        "pca_1": pca_1,
        "pca_2": pca_2,
        "pca_3": pca_3,

        "orientation": orientation,

        "height_width_ratio":
            height_width_ratio,

        "length_width_ratio":
            length_width_ratio,

        "height_length_ratio":
            height_length_ratio,

        "min_x": min_xyz[0],
        "min_y": min_xyz[1],
        "min_z": min_xyz[2],

        "max_x": max_xyz[0],
        "max_y": max_xyz[1],
        "max_z": max_xyz[2],
    }


# ============================================================
# 5. EXTRAER CLUSTERS Y FEATURES
# ============================================================

def extract_clusters(points, labels):

    cluster_rows = []
    cluster_points = {}

    unique_labels = np.unique(labels)

    for cluster_id in unique_labels:

        if cluster_id == -1:
            continue

        mask = labels == cluster_id
        cluster = points[mask]

        if len(cluster) == 0:
            continue

        cluster_points[cluster_id] = cluster

        # calculate_cluster_features solo recibe los puntos
        features = calculate_cluster_features(
            cluster
        )

        if features is not None:

            # Añadimos el ID del cluster aquí
            features["cluster_id"] = int(cluster_id)

            cluster_rows.append(features)

    if not cluster_rows:
        return pd.DataFrame(), cluster_points

    cluster_df = pd.DataFrame(
        cluster_rows
    )

    return cluster_df, cluster_points

# ============================================================
# 6. FILTRO GEOMÉTRICO
# ============================================================

def filter_candidate_clusters(
    features
):

    if features.empty:
        return features.copy()

    mask = (

        (features["height"] >= MIN_HEIGHT) &
        (features["height"] <= MAX_HEIGHT) &

        (features["length"] >= MIN_LENGTH) &
        (features["length"] <= MAX_LENGTH) &

        (features["width"] >= MIN_WIDTH) &
        (features["width"] <= MAX_WIDTH)
    )

    return (
        features[mask]
        .reset_index(drop=True)
    )


def get_frame_timestamp(sequence, frame_index):

    # timestamp principal del frame, en milisegundos
    timestamp_ms = sequence.time_sync_df.iloc[
        sequence.time_sync_df.index.get_loc("timestamp_ms"),
        frame_index
    ]

    timestamp_ms = float(timestamp_ms)

    # Convertir de milisegundos a segundos
    timestamp = timestamp_ms / 1000.0

    return timestamp


# ============================================================
# 7. OBTENER BOUNDING BOXES DEL GROUND TRUTH
# ============================================================

def get_vehicle_gt_boxes(
    gt_loader,
    timestamp
):

    vehicles = gt_loader.get_vehicle_objects(
        timestamp
    )

    boxes = []

    for vehicle in vehicles:

        boxes.append({
            "track_id": vehicle["track_id"],
            "position": np.asarray(
                vehicle["position"],
                dtype=np.float32
            ),
            "dimensions": np.asarray(
                vehicle["dimensions"],
                dtype=np.float32
            ),
            "orientation": float(
                vehicle["orientation"]
            ),
            "object_type": vehicle["object_type"]
        })

    return boxes


# ============================================================
# 8. COMPROBAR SI UN PUNTO ESTÁ DENTRO DE UNA BOX
# ============================================================

def points_inside_oriented_box(
    points,
    box_center,
    box_dimensions,
    yaw
):

    points = np.asarray(points)

    center = np.asarray(
        box_center
    )

    dimensions = np.asarray(
        box_dimensions
    )

    local = points - center

    c = np.cos(yaw)
    s = np.sin(yaw)

    x = local[:, 0]
    y = local[:, 1]

    local_x = c * x + s * y
    local_y = -s * x + c * y
    local_z = local[:, 2]

    inside = (
        (np.abs(local_x) <= dimensions[0] / 2)
        &
        (np.abs(local_y) <= dimensions[1] / 2)
        &
        (np.abs(local_z) <= dimensions[2] / 2)
    )

    return inside


# ============================================================
# 9. ETIQUETAR CLUSTERS CON GROUND TRUTH
# ============================================================

def label_clusters_with_gt(
    cluster_df,
    cluster_points,
    gt_boxes,
    overlap_threshold=0.25
):

    labels = []

    for cluster_id in cluster_df[
        "cluster_id"
    ]:

        points = cluster_points[
            cluster_id
        ]

        if points is None or len(points) == 0:

            labels.append(0)
            continue

        is_vehicle = False

        for box in gt_boxes:

            inside = points_inside_oriented_box(
                points,
                box["position"],
                box["dimensions"],
                box["orientation"]
            )

            overlap = np.mean(
                inside
            )

            if overlap >= overlap_threshold:

                is_vehicle = True
                break

        labels.append(
            1 if is_vehicle else 0
        )

    result = cluster_df.copy()

    result["label"] = labels

    return result


# ============================================================
# 10. PROCESAR UN FRAME
# ============================================================

def process_frame(
    points,
    eps=DBSCAN_EPS,
    min_samples=DBSCAN_MIN_SAMPLES
):

    labels = run_dbscan(
        points,
        eps=eps,
        min_samples=min_samples
    )

    if len(labels) == 0:
        return (
            pd.DataFrame(),
            {}
        )

    cluster_df, cluster_points = (
        extract_clusters(
            points,
            labels
        )
    )

    if cluster_df.empty:
        return (
            cluster_df,
            cluster_points
        )

    cluster_df = filter_candidate_clusters(
        cluster_df
    )

    # Importante:
    # conservar solo los puntos de los clusters
    # que han sobrevivido al filtrado.
    valid_ids = set(
        cluster_df["cluster_id"]
    )

    cluster_points = {
        cid: pts
        for cid, pts in cluster_points.items()
        if cid in valid_ids
    }

    return (
        cluster_df,
        cluster_points
    )


# ============================================================
# 11. LOCALIZAR FRAMES PREPROCESADOS
# ============================================================

def get_frame_directories(scenario):

    scenario_dir = os.path.join(
        PROCESSED_ROOT,
        scenario
    )

    if not os.path.exists(scenario_dir):
        return []

    frame_files = []

    for name in os.listdir(scenario_dir):

        if (
            name.startswith("frame_")
            and name.endswith("_processed.npy")
        ):

            path = os.path.join(
                scenario_dir,
                name
            )

            if os.path.isfile(path):
                frame_files.append(path)

    return sorted(frame_files)

# ============================================================
# 12. BUSCAR EL .NPY DE CADA FRAME
# ============================================================

def find_voxel_file(frame_file):

    if (
        isinstance(frame_file, str)
        and os.path.isfile(frame_file)
        and frame_file.endswith("_processed.npy")
    ):
        return frame_file

    return None

# ============================================================
# 13. CONSTRUIR DATASET DE CLUSTERS
# ============================================================

def build_cluster_dataset(
    scenarios,
    processed_root,
    root_folder
):

    all_frames = []

    for scenario in scenarios:

        print()
        print("=" * 80)
        print("SCENARIO:", scenario)
        print("=" * 80)

        sequence = SequenceNoGT(
            root_folder,
            scenario
        )

        gt_loader = UrbanIngGTLoader(
            root_folder,
            scenario
        )

        frame_dirs = get_frame_directories(
            scenario
        )

        print("Frames encontrados:", len(frame_dirs))

        # Contadores de diagnóstico
        n_voxel = 0
        n_empty_points = 0
        n_dbscan = 0
        n_empty_clusters = 0
        n_filtered = 0
        n_final = 0

        for frame_index, frame_dir in enumerate(frame_dirs):

            voxel_file = find_voxel_file(frame_dir)

            if voxel_file is None:
                continue

            n_voxel += 1

            points = np.load(voxel_file)

            if len(points) == 0:
                n_empty_points += 1
                continue

            # -------------------------------------------------
            # DBSCAN directamente
            # -------------------------------------------------

            labels = run_dbscan(
                points,
                eps=DBSCAN_EPS,
                min_samples=DBSCAN_MIN_SAMPLES
            )

            if len(labels) == 0:
                continue

            n_dbscan += 1

            unique_labels = np.unique(labels)
            n_clusters = np.sum(unique_labels != -1)

            # Mostrar primeros frames
            if frame_index < 5:
                print(
                    f"\nFrame {frame_index}: "
                    f"points={len(points)}, "
                    f"labels={len(labels)}, "
                    f"clusters={n_clusters}, "
                    f"noise={np.sum(labels == -1)}"
                )

            # -------------------------------------------------
            # Extraer clusters
            # -------------------------------------------------

            cluster_df, cluster_points = extract_clusters(
                points,
                labels
            )

            if cluster_df.empty:
                n_empty_clusters += 1

                if frame_index < 5:
                    print("  -> extract_clusters: VACÍO")

                continue

            # -------------------------------------------------
            # Filtrar candidatos
            # -------------------------------------------------

            before_filter = len(cluster_df)

            cluster_df = filter_candidate_clusters(
                cluster_df
            )

            after_filter = len(cluster_df)

            if frame_index < 5:
                print(
                    f"  clusters antes filtro: {before_filter}"
                )
                print(
                    f"  clusters después filtro: {after_filter}"
                )

            if cluster_df.empty:
                n_filtered += 1
                continue

            # Mantener solamente los puntos de los clusters
            # que sobrevivieron al filtro
            valid_ids = set(
                cluster_df["cluster_id"]
            )

            cluster_points = {
                cid: pts
                for cid, pts in cluster_points.items()
                if cid in valid_ids
            }

            # -------------------------------------------------
            # Timestamp
            # -------------------------------------------------

            timestamp = get_frame_timestamp(
                sequence,
                frame_index
            )

            # -------------------------------------------------
            # GT
            # -------------------------------------------------

            gt_boxes = get_vehicle_gt_boxes(
                gt_loader,
                timestamp
            )

            # -------------------------------------------------
            # Label
            # -------------------------------------------------

            cluster_df = label_clusters_with_gt(
                cluster_df,
                cluster_points,
                gt_boxes,
                overlap_threshold=0.25
            )

            cluster_df["scenario"] = scenario
            cluster_df["frame_index"] = frame_index
            cluster_df["timestamp"] = timestamp

            all_frames.append(cluster_df)

            n_final += 1

        print()
        print("RESUMEN DEL SCENARIO")
        print("--------------------")
        print("Voxel files:", n_voxel)
        print("Puntos vacíos:", n_empty_points)
        print("Frames con DBSCAN:", n_dbscan)
        print("Clusters vacíos:", n_empty_clusters)
        print("Frames sin candidatos:", n_filtered)
        print("Frames finales:", n_final)

    if not all_frames:

        raise RuntimeError(
            "No se generaron clusters. "
            "Revisa el resumen de diagnóstico anterior."
        )

    dataset = pd.concat(
        all_frames,
        ignore_index=True
    )

    return dataset

# ============================================================
# 14. CARGAR FRAME GT DE URBANING
# ============================================================

#
# Esta función utiliza tu SequenceNoGT para cargar el frame.
#
# Para el dataset de entrenamiento necesitamos una Sequence
# NORMAL porque aquí sí queremos acceder al Ground Truth.
#

from urbaning.data.sequence import Sequence


sequence_cache = {}


def load_gt_frame(
    scenario,
    frame_name
):

    if scenario not in sequence_cache:

        sequence_cache[scenario] = (
            Sequence(
                ROOT,
                scenario
            )
        )

    sequence = (
        sequence_cache[scenario]
    )

    # frame_0000 -> 0
    index = int(
        frame_name.split("_")[-1]
    )

    return sequence[index]


# ============================================================
# 15. CONSTRUIR DATASET
# ============================================================

# ------------------------------------------------------------
# IMPORTANTE
# ------------------------------------------------------------
#
# Esta línea puede tardar bastante porque:
#
# 1. lee todos los .npy
# 2. ejecuta DBSCAN
# 3. calcula features
# 4. carga el frame GT
# 5. etiqueta los clusters
#
# ------------------------------------------------------------

all_scenarios = (
    TRAIN_SCENARIOS
    + TEST_SCENARIOS
)

cluster_df = (
    build_cluster_dataset(
        scenarios=SCENARIOS,
        processed_root=PROCESSED_ROOT,
        root_folder=ROOT
    )
)


# ============================================================
# 16. COMPROBAR DATASET
# ============================================================

print()
print(
    cluster_df.shape
)

display(
    cluster_df.head()
)


# ============================================================
# 17. GUARDAR DATASET
# ============================================================

dataset_path = os.path.join(
    PROCESSED_ROOT,
    "dbscan_cluster_dataset.csv"
)

cluster_df.to_csv(
    dataset_path,
    index=False
)

print(
    "Dataset guardado en:"
)

print(
    dataset_path
)


# ============================================================
# 18. COMPROBAR QUE EXISTEN LAS ETIQUETAS
# ============================================================

if "label" not in cluster_df.columns:

    raise RuntimeError(
        "El dataset no contiene la columna 'label'. "
        "Revisa la función get_vehicle_gt_boxes()."
    )


print()
print(
    "Distribución global:"
)

print(
    cluster_df["label"]
    .value_counts()
)


# ============================================================
# 19. TRAIN / TEST POR ESCENARIO
# ============================================================

train_df = cluster_df[
    cluster_df["scenario"].isin(
        TRAIN_SCENARIOS
    )
].copy()

test_df = cluster_df[
    cluster_df["scenario"].isin(
        TEST_SCENARIOS
    )
].copy()


print()
print("=" * 70)
print("TRAIN / TEST")
print("=" * 70)

print(
    "Train:",
    len(train_df)
)

print(
    "Test:",
    len(test_df)
)


# ============================================================
# 20. VARIABLES X / y
# ============================================================

X_train = (
    train_df[
        FEATURE_COLUMNS
    ]
)

y_train = (
    train_df["label"]
)

X_test = (
    test_df[
        FEATURE_COLUMNS
    ]
)

y_test = (
    test_df["label"]
)


# ============================================================
# 21. RANDOM FOREST
# ============================================================

rf = RandomForestClassifier(

    n_estimators=N_ESTIMATORS,

    random_state=RANDOM_STATE,

    class_weight="balanced",

    n_jobs=-1
)


print()
print(
    "Entrenando Random Forest..."
)

rf.fit(
    X_train,
    y_train
)

print(
    "Entrenamiento terminado."
)


# ============================================================
# 22. PREDICCIÓN
# ============================================================

y_pred = rf.predict(
    X_test
)

y_probability = (
    rf.predict_proba(
        X_test
    )[:, 1]
)


# ============================================================
# 23. EVALUACIÓN
# ============================================================

print()
print("=" * 70)
print("EVALUACIÓN")
print("=" * 70)


accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)


print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1       : {f1:.4f}"
)


print()
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No vehículo",
            "Vehículo"
        ],
        zero_division=0
    )
)


# ============================================================
# 24. MATRIZ DE CONFUSIÓN
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print()
print(
    "Matriz de confusión:"
)

print(cm)


plt.figure(
    figsize=(6, 5)
)

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.title(
    "Matriz de confusión"
)

plt.xlabel(
    "Predicción"
)

plt.ylabel(
    "Real"
)

plt.xticks(
    [0, 1],
    ["No vehículo", "Vehículo"]
)

plt.yticks(
    [0, 1],
    ["No vehículo", "Vehículo"]
)

for i in range(2):

    for j in range(2):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()

plt.tight_layout()

plt.show()


# ============================================================
# 25. FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    "feature":
        FEATURE_COLUMNS,

    "importance":
        rf.feature_importances_

})

importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print()
print("=" * 70)
print("IMPORTANCIA DE FEATURES")
print("=" * 70)

display(
    importance_df
)


plt.figure(
    figsize=(10, 7)
)

plt.barh(
    importance_df["feature"],
    importance_df["importance"]
)

plt.xlabel(
    "Importancia"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Random Forest - Feature Importance"
)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.show()


# ============================================================
# 26. AÑADIR RESULTADOS AL TEST
# ============================================================

test_results = test_df.copy()

test_results[
    "prediction"
] = y_pred

test_results[
    "vehicle_probability"
] = y_probability

test_results[
    "correct"
] = (
    test_results["label"]
    ==
    test_results["prediction"]
)


# ============================================================
# 27. GUARDAR RESULTADOS
# ============================================================

results_path = os.path.join(
    PROCESSED_ROOT,
    "random_forest_test_results.csv"
)

test_results.to_csv(
    results_path,
    index=False
)

print()
print(
    "Resultados guardados en:"
)

print(
    results_path
)


# ============================================================
# 28. GUARDAR MODELO
# ============================================================

model_path = os.path.join(
    PROCESSED_ROOT,
    "random_forest_vehicle_detector.pkl"
)

joblib.dump(
    rf,
    model_path
)

print()
print(
    "Modelo guardado en:"
)

print(
    model_path
)


# ============================================================
# 29. FUNCIÓN PARA DETECTAR VEHÍCULOS EN UN FRAME NUEVO
# ============================================================

# ============================================================
# DETECCIÓN DE VEHÍCULOS
# DBSCAN + FILTRADO + RANDOM FOREST
# ============================================================

def detect_vehicles(points, model):

    # --------------------------------------------------------
    # 1. DBSCAN
    # --------------------------------------------------------

    labels = run_dbscan(
        points,
        eps=DBSCAN_EPS,
        min_samples=DBSCAN_MIN_SAMPLES
    )

    # --------------------------------------------------------
    # 2. Si no hay etiquetas
    # --------------------------------------------------------

    if len(labels) == 0:

        return (
            labels,
            pd.DataFrame(),
            {}
        )

    # --------------------------------------------------------
    # 3. Extraer clusters y puntos de cada cluster
    # --------------------------------------------------------

    cluster_df, cluster_points = extract_clusters(
        points,
        labels
    )

    # --------------------------------------------------------
    # 4. No hay clusters
    # --------------------------------------------------------

    if cluster_df.empty:

        return (
            labels,
            cluster_df,
            cluster_points
        )

    # --------------------------------------------------------
    # 5. Filtrar clusters candidatos
    # --------------------------------------------------------

    cluster_df = filter_candidate_clusters(
        cluster_df
    )

    # --------------------------------------------------------
    # 6. No quedan clusters después del filtro
    # --------------------------------------------------------

    if cluster_df.empty:

        return (
            labels,
            cluster_df,
            cluster_points
        )

    # --------------------------------------------------------
    # 7. Mantener solamente los puntos de los clusters
    #    que sobrevivieron al filtrado
    # --------------------------------------------------------

    valid_ids = set(
        cluster_df["cluster_id"]
    )

    cluster_points = {
        cid: pts
        for cid, pts in cluster_points.items()
        if cid in valid_ids
    }

    # --------------------------------------------------------
    # 8. Preparar características para Random Forest
    # --------------------------------------------------------

    X = cluster_df[
        FEATURE_COLUMNS
    ]

    # --------------------------------------------------------
    # 9. Predicción
    # --------------------------------------------------------

    predictions = model.predict(
        X
    )

    probabilities = model.predict_proba(
        X
    )[:, 1]

    # --------------------------------------------------------
    # 10. Añadir resultados al DataFrame
    # --------------------------------------------------------

    cluster_df = cluster_df.copy()

    cluster_df["prediction"] = predictions

    cluster_df["vehicle_probability"] = probabilities

    # --------------------------------------------------------
    # 11. Devolver resultados
    # --------------------------------------------------------

    return (
        labels,
        cluster_df,
        cluster_points
    )

# ============================================================
# 30. VISUALIZAR VEHÍCULOS DETECTADOS
# ============================================================

def plot_detected_vehicles(
    points,
    labels,
    predictions
):

    plt.figure(
        figsize=(12, 10)
    )

    # --------------------------------------------------------
    # Nube completa
    # --------------------------------------------------------

    plt.scatter(
        points[:, 0],
        points[:, 1],
        s=1,
        alpha=0.08
    )

    # --------------------------------------------------------
    # Vehículos detectados
    # --------------------------------------------------------

    for _, row in (
        predictions.iterrows()
    ):

        if (
            row["vehicle_prediction"]
            != 1
        ):

            continue

        cluster_id = int(
            row["cluster_id"]
        )

        mask = (
            labels
            == cluster_id
        )

        cluster_points = (
            points[mask]
        )

        # Cluster
        plt.scatter(
            cluster_points[:, 0],
            cluster_points[:, 1],
            s=8,
            alpha=0.8
        )

        # Centroide
        plt.scatter(
            row["centroid_x"],
            row["centroid_y"],
            s=100,
            marker="x"
        )

        # ID
        plt.text(
            row["centroid_x"],
            row["centroid_y"],
            f" {cluster_id}",
            fontsize=9
        )

    plt.xlabel(
        "X [m]"
    )

    plt.ylabel(
        "Y [m]"
    )

    plt.title(
        "Vehículos detectados mediante "
        "DBSCAN + Random Forest"
    )

    plt.axis(
        "equal"
    )

    plt.grid(
        True
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 31. EJEMPLO DE INFERENCIA
# ============================================================

#
# Seleccionamos el primer frame disponible del TEST.
#

test_scenario = TEST_SCENARIOS[0]

test_frame_dirs = (
    get_frame_directories(
        test_scenario
    )
)

if len(test_frame_dirs) > 0:

    example_frame_dir = (
        test_frame_dirs[0]
    )

    voxel_file = (
        find_voxel_file(
            example_frame_dir
        )
    )

    if voxel_file is not None:

        example_points = np.load(
            voxel_file
        )

        example_points = (
            example_points[:, :3]
        )

        (
            example_labels,
            example_predictions,
            example_cluster_points
        ) = detect_vehicles(
            example_points,
            rf
        )

        print()
        print("=" * 70)
        print("EJEMPLO DE DETECCIÓN")
        print("=" * 70)

        print(
            "Clusters:",
            len(
                np.unique(
                    example_labels[
                        example_labels >= 0
                    ]
                )
            )
        )

        print(
            "Candidatos:",
            len(
                example_predictions
            )
        )

        if not example_predictions.empty:

            print(
                "Vehículos detectados:",
                int(
                    example_predictions[
                        "vehicle_prediction"
                    ].sum()
                )
            )

            display(
                example_predictions[
                    [
                        "cluster_id",
                        "n_points",
                        "length",
                        "width",
                        "height",
                        "centroid_x",
                        "centroid_y",
                        "vehicle_probability",
                        "vehicle_prediction"
                    ]
                ]
            )

            plot_detected_vehicles(
                example_points,
                example_labels,
                example_predictions
            )


SCENARIO: 20241126_0008_crossing1_01
Frames encontrados: 200

Frame 0: points=66880, labels=66880, clusters=163, noise=307
  clusters antes filtro: 163
  clusters después filtro: 52

Frame 1: points=69714, labels=69714, clusters=129, noise=252
  clusters antes filtro: 129
  clusters después filtro: 51

Frame 2: points=83881, labels=83881, clusters=133, noise=227
  clusters antes filtro: 133
  clusters después filtro: 47

Frame 3: points=72437, labels=72437, clusters=153, noise=270
  clusters antes filtro: 153
  clusters después filtro: 51

Frame 4: points=70482, labels=70482, clusters=152, noise=188
  clusters antes filtro: 152
  clusters después filtro: 52


KeyboardInterrupt: 

In [35]:
# ============================================================
# DETECCIÓN DE VEHÍCULOS
# DBSCAN + FILTRADO + RANDOM FOREST
# ============================================================

def detect_vehicles(points, model):

    # --------------------------------------------------------
    # 1. DBSCAN
    # --------------------------------------------------------

    labels = run_dbscan(
        points,
        eps=DBSCAN_EPS,
        min_samples=DBSCAN_MIN_SAMPLES
    )

    # --------------------------------------------------------
    # 2. Si no hay etiquetas
    # --------------------------------------------------------

    if len(labels) == 0:

        return (
            labels,
            pd.DataFrame(),
            {}
        )

    # --------------------------------------------------------
    # 3. Extraer clusters y puntos de cada cluster
    # --------------------------------------------------------

    cluster_df, cluster_points = extract_clusters(
        points,
        labels
    )

    # --------------------------------------------------------
    # 4. No hay clusters
    # --------------------------------------------------------

    if cluster_df.empty:

        return (
            labels,
            cluster_df,
            cluster_points
        )

    # --------------------------------------------------------
    # 5. Filtrar clusters candidatos
    # --------------------------------------------------------

    cluster_df = filter_candidate_clusters(
        cluster_df
    )

    # --------------------------------------------------------
    # 6. No quedan clusters después del filtro
    # --------------------------------------------------------

    if cluster_df.empty:

        return (
            labels,
            cluster_df,
            cluster_points
        )

    # --------------------------------------------------------
    # 7. Mantener solamente los puntos de los clusters
    #    que sobrevivieron al filtrado
    # --------------------------------------------------------

    valid_ids = set(
        cluster_df["cluster_id"]
    )

    cluster_points = {
        cid: pts
        for cid, pts in cluster_points.items()
        if cid in valid_ids
    }

    # --------------------------------------------------------
    # 8. Preparar características para Random Forest
    # --------------------------------------------------------

    X = cluster_df[
        FEATURE_COLUMNS
    ]

    # --------------------------------------------------------
    # 9. Predicción
    # --------------------------------------------------------

    predictions = model.predict(
        X
    )

    probabilities = model.predict_proba(
        X
    )[:, 1]

    # --------------------------------------------------------
    # 10. Añadir resultados al DataFrame
    # --------------------------------------------------------

    cluster_df = cluster_df.copy()

    cluster_df["prediction"] = predictions

    cluster_df["vehicle_probability"] = probabilities

    # --------------------------------------------------------
    # 11. Devolver resultados
    # --------------------------------------------------------

    return (
        labels,
        cluster_df,
        cluster_points
    )

In [39]:
# ============================================================
# VIDEO BEV
# DBSCAN + RANDOM FOREST + TRACKER TEMPORAL
# RENDERIZADO DIRECTO CON OPENCV
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

VIDEO_SCENARIO = TEST_SCENARIOS[0]

VIDEO_OUTPUT_DIR = os.path.join(
    PROCESSED_ROOT,
    "videos"
)

os.makedirs(
    VIDEO_OUTPUT_DIR,
    exist_ok=True
)

VIDEO_OUTPUT = os.path.join(
    VIDEO_OUTPUT_DIR,
    f"{VIDEO_SCENARIO}_RF_BEV_TRACKED.mp4"
)

FPS = 10

# Límites del BEV
X_MIN = -40
X_MAX = 40

Y_MIN = -50
Y_MAX = 20

# Resolución del vídeo
VIDEO_WIDTH = 1200
VIDEO_HEIGHT = 900

# Margen del gráfico
MARGIN_LEFT = 80
MARGIN_RIGHT = 30
MARGIN_TOP = 60
MARGIN_BOTTOM = 70

# Umbral Random Forest
VEHICLE_THRESHOLD_VIDEO = 0.5


# ============================================================
# 2. TRACKER
# ============================================================

TRACK_MAX_DISTANCE = 3.0
TRACK_MAX_MISSED = 5
TRACK_SMOOTHING = 0.5
TRACK_MIN_HITS = 2


class DistanceTracker:

    def __init__(
        self,
        max_distance=3.0,
        max_missed=5,
        smoothing=0.5,
        min_hits=2
    ):

        self.max_distance = max_distance
        self.max_missed = max_missed
        self.smoothing = smoothing
        self.min_hits = min_hits

        self.tracks = {}

        self.next_track_id = 0


    def _create_track(self, detection):

        track_id = self.next_track_id

        self.next_track_id += 1

        self.tracks[track_id] = {

            "track_id": track_id,

            "centroid_x": float(
                detection["centroid_x"]
            ),

            "centroid_y": float(
                detection["centroid_y"]
            ),

            "length": float(
                detection["length"]
            ),

            "width": float(
                detection["width"]
            ),

            "height": float(
                detection["height"]
            ),

            "orientation": float(
                detection["orientation"]
            ),

            "probability": float(
                detection["vehicle_probability"]
            ),

            "hits": 1,

            "missed": 0,

            "age": 1
        }


    def _smooth_angle(
        self,
        old_angle,
        new_angle
    ):

        delta = np.arctan2(
            np.sin(new_angle - old_angle),
            np.cos(new_angle - old_angle)
        )

        return (
            old_angle
            + self.smoothing * delta
        )


    def _update_track(
        self,
        track,
        detection
    ):

        alpha = self.smoothing

        track["centroid_x"] = (
            (1 - alpha)
            * track["centroid_x"]
            + alpha
            * float(detection["centroid_x"])
        )

        track["centroid_y"] = (
            (1 - alpha)
            * track["centroid_y"]
            + alpha
            * float(detection["centroid_y"])
        )

        track["length"] = (
            (1 - alpha)
            * track["length"]
            + alpha
            * float(detection["length"])
        )

        track["width"] = (
            (1 - alpha)
            * track["width"]
            + alpha
            * float(detection["width"])
        )

        track["height"] = (
            (1 - alpha)
            * track["height"]
            + alpha
            * float(detection["height"])
        )

        track["orientation"] = (
            self._smooth_angle(
                track["orientation"],
                float(detection["orientation"])
            )
        )

        track["probability"] = (
            (1 - alpha)
            * track["probability"]
            + alpha
            * float(detection["vehicle_probability"])
        )

        track["hits"] += 1
        track["missed"] = 0
        track["age"] += 1


    def _remove_old_tracks(self):

        to_remove = []

        for track_id, track in self.tracks.items():

            if track["missed"] > self.max_missed:

                to_remove.append(track_id)

        for track_id in to_remove:

            del self.tracks[track_id]


    def update(self, detections):

        # ----------------------------------------------------
        # No detections
        # ----------------------------------------------------

        if (
            detections is None
            or detections.empty
        ):

            for track in self.tracks.values():

                track["missed"] += 1
                track["age"] += 1

            self._remove_old_tracks()

            return self.get_active_tracks()


        detection_list = [
            row
            for _, row in detections.iterrows()
        ]


        # ----------------------------------------------------
        # No tracks todavía
        # ----------------------------------------------------

        if len(self.tracks) == 0:

            for detection in detection_list:

                self._create_track(
                    detection
                )

            return self.get_active_tracks()


        # ----------------------------------------------------
        # Calcular distancias
        # ----------------------------------------------------

        candidates = []

        for track_id, track in self.tracks.items():

            tx = track["centroid_x"]
            ty = track["centroid_y"]

            for det_idx, detection in enumerate(
                detection_list
            ):

                dx = float(
                    detection["centroid_x"]
                )

                dy = float(
                    detection["centroid_y"]
                )

                distance = np.sqrt(
                    (tx - dx) ** 2
                    + (ty - dy) ** 2
                )

                candidates.append(
                    (
                        distance,
                        track_id,
                        det_idx
                    )
                )


        candidates.sort(
            key=lambda x: x[0]
        )


        matched_tracks = set()
        matched_detections = set()


        # ----------------------------------------------------
        # Asociación greedy
        # ----------------------------------------------------

        for (
            distance,
            track_id,
            det_idx
        ) in candidates:

            if distance > self.max_distance:
                break

            if track_id in matched_tracks:
                continue

            if det_idx in matched_detections:
                continue

            self._update_track(
                self.tracks[track_id],
                detection_list[det_idx]
            )

            matched_tracks.add(
                track_id
            )

            matched_detections.add(
                det_idx
            )


        # ----------------------------------------------------
        # Tracks no asociados
        # ----------------------------------------------------

        for track_id in self.tracks:

            if track_id not in matched_tracks:

                self.tracks[
                    track_id
                ]["missed"] += 1

                self.tracks[
                    track_id
                ]["age"] += 1


        # ----------------------------------------------------
        # Crear nuevos tracks
        # ----------------------------------------------------

        for det_idx, detection in enumerate(
            detection_list
        ):

            if det_idx not in matched_detections:

                self._create_track(
                    detection
                )


        self._remove_old_tracks()

        return self.get_active_tracks()


    def get_active_tracks(self):

        active = []

        for track in self.tracks.values():

            if (
                track["hits"] < self.min_hits
                and track["missed"] > 0
            ):
                continue

            if track["missed"] > self.max_missed:
                continue

            active.append(
                track.copy()
            )

        return active


# ============================================================
# 3. CONVERSIÓN COORDENADAS → PIXEL
# ============================================================

def world_to_pixel(x, y):

    plot_width = (
        VIDEO_WIDTH
        - MARGIN_LEFT
        - MARGIN_RIGHT
    )

    plot_height = (
        VIDEO_HEIGHT
        - MARGIN_TOP
        - MARGIN_BOTTOM
    )

    px = (
        MARGIN_LEFT
        + (
            (x - X_MIN)
            / (X_MAX - X_MIN)
        )
        * plot_width
    )

    py = (
        MARGIN_TOP
        + (
            (Y_MAX - y)
            / (Y_MAX - Y_MIN)
        )
        * plot_height
    )

    return (
        int(round(px)),
        int(round(py))
    )


# ============================================================
# 4. CAJA ORIENTADA
# ============================================================

def get_box_pixels(
    x,
    y,
    length,
    width,
    orientation
):

    half_l = length / 2
    half_w = width / 2

    corners = np.array([
        [ half_l,  half_w],
        [ half_l, -half_w],
        [-half_l, -half_w],
        [-half_l,  half_w]
    ], dtype=np.float32)

    c = np.cos(
        orientation
    )

    s = np.sin(
        orientation
    )

    rotation = np.array([
        [c, -s],
        [s,  c]
    ], dtype=np.float32)

    corners = corners @ rotation.T

    corners[:, 0] += x
    corners[:, 1] += y

    pixels = []

    for px, py in corners:

        pixels.append(
            world_to_pixel(
                px,
                py
            )
        )

    return np.asarray(
        pixels,
        dtype=np.int32
    )


# ============================================================
# 5. CREAR FRAME BEV DIRECTAMENTE CON OPENCV
# ============================================================

def create_bev_frame_cv(
    points,
    tracks,
    frame_index,
    scenario
):

    # --------------------------------------------------------
    # Fondo
    # --------------------------------------------------------

    image = np.ones(
        (
            VIDEO_HEIGHT,
            VIDEO_WIDTH,
            3
        ),
        dtype=np.uint8
    ) * 255


    # --------------------------------------------------------
    # Área del BEV
    # --------------------------------------------------------

    cv2.rectangle(
        image,
        (
            MARGIN_LEFT,
            MARGIN_TOP
        ),
        (
            VIDEO_WIDTH - MARGIN_RIGHT,
            VIDEO_HEIGHT - MARGIN_BOTTOM
        ),
        (245, 245, 245),
        -1
    )


    # --------------------------------------------------------
    # Dibujar rejilla
    # --------------------------------------------------------

    for x in np.arange(
        X_MIN,
        X_MAX + 1,
        5
    ):

        px1, py1 = world_to_pixel(
            x,
            Y_MIN
        )

        px2, py2 = world_to_pixel(
            x,
            Y_MAX
        )

        cv2.line(
            image,
            (px1, py1),
            (px2, py2),
            (220, 220, 220),
            1
        )


    for y in np.arange(
        Y_MIN,
        Y_MAX + 1,
        5
    ):

        px1, py1 = world_to_pixel(
            X_MIN,
            y
        )

        px2, py2 = world_to_pixel(
            X_MAX,
            y
        )

        cv2.line(
            image,
            (px1, py1),
            (px2, py2),
            (220, 220, 220),
            1
        )


    # --------------------------------------------------------
    # LiDAR
    # --------------------------------------------------------

    n_points_drawn = 0

    if points is not None and len(points) > 0:

        points = np.asarray(
            points
        )

        mask = (
            np.isfinite(
                points[:, :3]
            ).all(axis=1)
            &
            (points[:, 0] >= X_MIN)
            &
            (points[:, 0] <= X_MAX)
            &
            (points[:, 1] >= Y_MIN)
            &
            (points[:, 1] <= Y_MAX)
        )

        visible = points[
            mask
        ]

        # ----------------------------------------------------
        # Dibujar puntos
        # ----------------------------------------------------

        for point in visible:

            px, py = world_to_pixel(
                float(point[0]),
                float(point[1])
            )

            cv2.circle(
                image,
                (px, py),
                1,
                (100, 100, 100),
                -1
            )

        n_points_drawn = len(
            visible
        )


    # --------------------------------------------------------
    # Origen
    # --------------------------------------------------------

    ox, oy = world_to_pixel(
        0,
        0
    )

    cv2.drawMarker(
        image,
        (ox, oy),
        (0, 0, 0),
        markerType=cv2.MARKER_CROSS,
        markerSize=20,
        thickness=2
    )


    # --------------------------------------------------------
    # Vehículos
    # --------------------------------------------------------

    n_tracks = 0

    for track in tracks:

        x = float(
            track["centroid_x"]
        )

        y = float(
            track["centroid_y"]
        )

        length = float(
            track["length"]
        )

        width = float(
            track["width"]
        )

        orientation = float(
            track["orientation"]
        )

        probability = float(
            track["probability"]
        )

        track_id = int(
            track["track_id"]
        )

        missed = int(
            track["missed"]
        )


        # ----------------------------------------------------
        # Caja
        # ----------------------------------------------------

        corners = get_box_pixels(
            x,
            y,
            length,
            width,
            orientation
        )

        corners_closed = np.vstack([
            corners,
            corners[0]
        ])

        # BGR
        box_color = (
            0,
            0,
            255
        )

        cv2.polylines(
            image,
            [corners_closed],
            False,
            box_color,
            3
        )


        # ----------------------------------------------------
        # Centro
        # ----------------------------------------------------

        cx, cy = world_to_pixel(
            x,
            y
        )

        cv2.circle(
            image,
            (cx, cy),
            5,
            (0, 0, 255),
            -1
        )


        # ----------------------------------------------------
        # Texto
        # ----------------------------------------------------

        text = (
            f"ID {track_id} "
            f"P={probability:.2f}"
        )

        if missed > 0:

            text += (
                f"  lost={missed}"
            )

        cv2.putText(
            image,
            text,
            (
                cx + 8,
                cy - 8
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 180),
            2,
            cv2.LINE_AA
        )

        n_tracks += 1


    # --------------------------------------------------------
    # Ejes / etiquetas
    # --------------------------------------------------------

    cv2.putText(
        image,
        "X [m]",
        (
            VIDEO_WIDTH // 2,
            VIDEO_HEIGHT - 20
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    cv2.putText(
        image,
        "Y [m]",
        (
            15,
            VIDEO_HEIGHT // 2
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    # --------------------------------------------------------
    # Título
    # --------------------------------------------------------

    title = (
        f"{scenario} | "
        f"Frame {frame_index:04d} | "
        f"Tracks: {n_tracks}"
    )

    cv2.putText(
        image,
        title,
        (
            MARGIN_LEFT,
            35
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    # --------------------------------------------------------
    # Información LiDAR
    # --------------------------------------------------------

    info = (
        f"LiDAR points: {n_points_drawn}"
    )

    cv2.putText(
        image,
        info,
        (
            MARGIN_LEFT,
            VIDEO_HEIGHT - 30
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (80, 80, 80),
        1,
        cv2.LINE_AA
    )


    return image


# ============================================================
# 6. GENERAR VÍDEO
# ============================================================

def generate_bev_video(
    scenario,
    model,
    output_file,
    fps=10
):

    print("=" * 70)
    print("GENERANDO VIDEO BEV + TRACKER")
    print("=" * 70)

    print(
        f"Escenario: {scenario}"
    )

    print(
        f"Salida: {output_file}"
    )

    print(
        f"FPS: {fps}"
    )

    print()


    # --------------------------------------------------------
    # Obtener frames
    # --------------------------------------------------------

    frame_files = get_frame_files_video(
        scenario
    )

    if len(frame_files) == 0:

        raise RuntimeError(
            "No se encontraron frames."
        )

    print(
        f"Frames encontrados: "
        f"{len(frame_files)}"
    )


    # --------------------------------------------------------
    # Tracker
    # --------------------------------------------------------

    tracker = DistanceTracker(
        max_distance=TRACK_MAX_DISTANCE,
        max_missed=TRACK_MAX_MISSED,
        smoothing=TRACK_SMOOTHING,
        min_hits=TRACK_MIN_HITS
    )


    # --------------------------------------------------------
    # VideoWriter
    # --------------------------------------------------------

    fourcc = cv2.VideoWriter_fourcc(
        *"mp4v"
    )

    writer = cv2.VideoWriter(
        output_file,
        fourcc,
        fps,
        (
            VIDEO_WIDTH,
            VIDEO_HEIGHT
        )
    )

    if not writer.isOpened():

        raise RuntimeError(
            "No se pudo abrir VideoWriter."
        )


    # ========================================================
    # PROCESAR
    # ========================================================

    total = len(
        frame_files
    )

    try:

        for i, frame_file in enumerate(
            frame_files
        ):

            # ------------------------------------------------
            # Cargar puntos
            # ------------------------------------------------

            points = np.load(
                frame_file
            )


            # ------------------------------------------------
            # DBSCAN + RF
            # ------------------------------------------------

            (
                labels,
                detections,
                cluster_points
            ) = detect_vehicles(
                points,
                model
            )


            # ------------------------------------------------
            # Filtrar vehículos
            # ------------------------------------------------

            if (
                detections is not None
                and not detections.empty
            ):

                detections_vehicle = (
                    detections[
                        detections["prediction"] == 1
                    ]
                    .copy()
                )

                detections_vehicle = (
                    detections_vehicle[
                        detections_vehicle[
                            "vehicle_probability"
                        ]
                        >= VEHICLE_THRESHOLD_VIDEO
                    ]
                    .copy()
                )

            else:

                detections_vehicle = (
                    pd.DataFrame()
                )


            # ------------------------------------------------
            # TRACKER
            # ------------------------------------------------

            tracks = tracker.update(
                detections_vehicle
            )


            # ------------------------------------------------
            # Crear frame
            # ------------------------------------------------

            frame_image = create_bev_frame_cv(
                points=points,
                tracks=tracks,
                frame_index=i,
                scenario=scenario
            )


            # ------------------------------------------------
            # Escribir
            # ------------------------------------------------

            writer.write(
                frame_image
            )


            # ------------------------------------------------
            # Progreso
            # ------------------------------------------------

            if (
                i == 0
                or (i + 1) % 10 == 0
                or i == total - 1
            ):

                print(
                    f"Frame "
                    f"{i + 1:03d}/{total:03d} | "
                    f"RF: "
                    f"{len(detections_vehicle):02d} | "
                    f"Tracks: "
                    f"{len(tracks):02d}"
                )

    finally:

        writer.release()

        cv2.destroyAllWindows()


    # --------------------------------------------------------
    # Comprobar archivo
    # --------------------------------------------------------

    if not os.path.exists(
        output_file
    ):

        raise RuntimeError(
            "El archivo de vídeo no se ha creado."
        )


    file_size = os.path.getsize(
        output_file
    )

    print()
    print("=" * 70)
    print("VIDEO GENERADO")
    print("=" * 70)

    print(
        f"Archivo:\n{output_file}"
    )

    print(
        f"Tamaño: "
        f"{file_size / (1024 * 1024):.2f} MB"
    )


# ============================================================
# 7. EJECUTAR
# ============================================================

generate_bev_video(
    scenario=VIDEO_SCENARIO,
    model=rf,
    output_file=VIDEO_OUTPUT,
    fps=FPS
)

GENERANDO VIDEO BEV + TRACKER
Escenario: 20241127_0000_crossing1_00
Salida: datasets/UrbanIng-V2X\processed_lidar\videos\20241127_0000_crossing1_00_RF_BEV_TRACKED.mp4
FPS: 10

Frames encontrados: 200
Frame 001/200 | RF: 18 | Tracks: 18
Frame 010/200 | RF: 18 | Tracks: 28
Frame 020/200 | RF: 15 | Tracks: 23
Frame 030/200 | RF: 13 | Tracks: 19
Frame 040/200 | RF: 10 | Tracks: 18
Frame 050/200 | RF: 16 | Tracks: 22
Frame 060/200 | RF: 18 | Tracks: 23
Frame 070/200 | RF: 18 | Tracks: 26
Frame 080/200 | RF: 17 | Tracks: 27
Frame 090/200 | RF: 16 | Tracks: 25
Frame 100/200 | RF: 14 | Tracks: 24
Frame 110/200 | RF: 16 | Tracks: 27
Frame 120/200 | RF: 17 | Tracks: 29
Frame 130/200 | RF: 18 | Tracks: 30
Frame 140/200 | RF: 14 | Tracks: 30
Frame 150/200 | RF: 10 | Tracks: 29
Frame 160/200 | RF: 19 | Tracks: 35
Frame 170/200 | RF: 18 | Tracks: 33
Frame 180/200 | RF: 12 | Tracks: 30
Frame 190/200 | RF: 15 | Tracks: 32
Frame 200/200 | RF: 18 | Tracks: 28

VIDEO GENERADO
Archivo:
datasets/UrbanIng-V